> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [Foundry IQ概要](#foundry-iq概要)
- [AI Search接続](#ai-search接続)
- [Knowledge Base作成 (AI Search Index)](#knowledge-base作成-ai-search-index)
- [Knowledge Base作成 (Blob Storage)](#knowledge-base作成-blob-storage)
- [KnowledgeAgent統合](#knowledgeagent統合)

## 🎯 学習目標

- Foundry IQの概念とメリットの理解
- Azure AI Searchリソースの接続と構成
- AI Search IndexベースのKnowledge Base作成
- Blob StorageベースのKnowledge Base
- Knowledge Baseをエージェントに統合する方法の学習

## ⏱️ 予想所要時間

約40分

## Foundry IQ概要

### Foundry IQとは?

Foundry IQは Microsoft Foundryの インテリジェント 知識 管理 システムで, 各種 データ ソースを 統合して AI エージェントに 文脈的 知識を 提供します.

### 主要 特徴

```
Foundry IQ = Retrieval + Reasoning + Ranking
```

- **Retrieval**: 関連 情報を 効率的で 検索
- **Reasoning**: 検索された 情報を が理解し 解釈
- **Ranking**: が章 関連性 高は 情報を 優先順上化

## 環境設定

Knowledge Base 構築を 上した Azure リソースを 設定します.

### Azure 環境変数 および パッケージ ロード

In [ ]:
# 環境変数 ロード
import json
import os
import subprocess

# PATH 環境変数 設定 (Azure CLIを 見つを 数 あるも録)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# 前 ノートブックで 保存した 設定ファイル ロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境変数でも 設定 (異なる もそれらが 使用する 数 あるも録)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定ファイル '{config_file}'で 環境変数を ロードしました。")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイルを 見つを 数 ありません。")
    print("💡 01-setup.ipynbを まず 実行して 環境を 設定してください.")
    raise

# 必須 パッケージ インストール
%pip install -q azure-ai-projects azure-identity azure-search-documents requests

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# Project Client 初期化
credential = DefaultAzureCredential()

# プロジェクト 名前 抽出 (URLで 最後 部分)
# はい: https://foundry-xxx.services.ai.azure.com/api/projects/default-project
# → project_name = "default-project"
import re
match = re.search(r'/projects/([^/]+)$', PROJECT_ENDPOINT)
if match:
    project_name = match.group(1)
else:
    project_name = PROJECT_NAME  # fallback to config

# Foundry URL (プロジェクト 部分 除外)
foundry_base_url = PROJECT_ENDPOINT.rsplit('/projects/', 1)[0]

project_client = AIProjectClient(
    endpoint=foundry_base_url,
    credential=credential,
    project_name=project_name
)

print(f"\n💡 使用する プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
print(f"✅ Project Client 初期化 完了")
print(f"   Foundry: {foundry_base_url}")
print(f"   Project: {project_name}")

### AI Search および Storage リソース 名前 作成

In [ ]:
# AI Search および Storage リソース 名前 設定
import re

# ユニークな 名前 作成 (小文字, 数字, するがオープンだけ 許可)
def sanitize_name(name, max_length=24):
    """リソース 名前を Azure ルールに に合わせて 整理"""
    # 小文字で 変換し 英数字だけ 残す
    clean = re.sub(r'[^a-z0-9]', '', name.lower())
    return clean[:max_length]

# FOUNDRY_NAME ベースで ユニーク 名前 作成
base_name = sanitize_name(FOUNDRY_NAME)

SEARCH_NAME = f"{base_name}-search"[:64]  # AI Searchは 最大 64者
STORAGE_NAME = sanitize_name(base_name + "store", 24)  # Storageは 最大 24者, するがオープン 不が
SEARCH_INDEX_NAME = "knowledge-index"

print("📌 作成する リソース 名前:")
print(f"   AI Search: {SEARCH_NAME}")
print(f"   Storage Account: {STORAGE_NAME}")
print(f"   Search Index: {SEARCH_INDEX_NAME}")

# 設定ファイルに 保存
config["SEARCH_NAME"] = SEARCH_NAME
config["STORAGE_NAME"] = STORAGE_NAME
config["SEARCH_INDEX_NAME"] = SEARCH_INDEX_NAME

with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ リソース 名前が '{config_file}'に 保存なりました。")

## Azure AI Search 作成

ベクトル 検索を 上した AI Search リソースを 作成します.

### AI Searchリソースの作成 実行

In [ ]:
# AI Search リソース 作成
!az search service create \
    --name $SEARCH_NAME \
    --resource-group $RESOURCE_GROUP \
    --location $LOCATION \
    --sku basic

print(f"\n✅ AI Search 作成 完了: {SEARCH_NAME}")

## Blob Storage 作成

ドキュメント ファイルを 保存する Blob Storageを 作成します.

### Storage Account および Container 作成

In [ ]:
# Storage Account 作成
!az storage account create \
    --name $STORAGE_NAME \
    --resource-group $RESOURCE_GROUP \
    --location $LOCATION \
    --sku Standard_LRS \
    --public-network-access Enabled

print(f"\n✅ Storage Account 作成 完了: {STORAGE_NAME}")

# Storage Container 作成
!az storage container create \
    --name documents \
    --account-name $STORAGE_NAME \
    --auth-mode login

print(f"✅ Container 'documents' 作成 完了!")

## Managed Identity 有効化

AI Searchが Storageと Foundryに アクセスする 数 あるも録 Managed Identityを 設定します.

In [ ]:
# AI Searchの Managed Identity 有効化
!az search service update \
    --name $SEARCH_NAME \
    --resource-group $RESOURCE_GROUP \
    --identity-type SystemAssigned

print(f"\n✅ AI Search Managed Identity 有効化 完了")

## IAM権限の設定

Storage Accountと AI Search, Foundry 間の 権限を 設定します.

### ユーザー および リソース 情報 取得

In [ ]:
# 現在 ユーザー 情報 が取得
import subprocess
import json

# 現在 ログによる ユーザー Object ID
result = subprocess.run(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"], 
                       capture_output=True, text=True)
USER_OBJECT_ID = result.stdout.strip()
print(f"📌 現在 ユーザー Object ID: {USER_OBJECT_ID}")

# AI Searchの Principal ID が取得
result = subprocess.run([
    "az", "search", "service", "show",
    "--name", SEARCH_NAME,
    "--resource-group", RESOURCE_GROUP,
    "--query", "identity.principalId", "-o", "tsv"
], capture_output=True, text=True)
SEARCH_PRINCIPAL_ID = result.stdout.strip()
print(f"📌 AI Search Principal ID: {SEARCH_PRINCIPAL_ID}")

# Storage Account Resource ID が取得
result = subprocess.run([
    "az", "storage", "account", "show",
    "--name", STORAGE_NAME,
    "--resource-group", RESOURCE_GROUP,
    "--query", "id", "-o", "tsv"
], capture_output=True, text=True)
STORAGE_RESOURCE_ID = result.stdout.strip()
print(f"📌 Storage Resource ID: {STORAGE_RESOURCE_ID}")

### Storage Blob Data Contributor ロール 割り当て (ユーザー & AI Search MI)

In [ ]:
# 1. Storage Blob Data Contributor - 現在 ユーザー
print("1️⃣ Storage Blob Data Contributor ロール 割り当て (現在 ユーザー)...")
!az role assignment create \
    --role "Storage Blob Data Contributor" \
    --assignee $USER_OBJECT_ID \
    --scope $STORAGE_RESOURCE_ID

# 2. Storage Blob Data Contributor - AI Search (Managed Identity)
print("\n2️⃣ Storage Blob Data Contributor ロール 割り当て (AI Search)...")
!az role assignment create \
    --role "Storage Blob Data Contributor" \
    --assignee $SEARCH_PRINCIPAL_ID \
    --scope $STORAGE_RESOURCE_ID

print("\n✅ Storage IAM 権限 設定 完了!")

### Azure AI Project Manager ロール 割り当て (AI Search MI)

In [ ]:
# 3. Foundry リソースに Azure AI Project Manager ロール 割り当て (AI Search MI)
import subprocess
import os

# Foundry Resource ID が取得
result = subprocess.run([
    "az", "cognitiveservices", "account", "show",
    "--name", FOUNDRY_NAME,
    "--resource-group", RESOURCE_GROUP,
    "--query", "id", "-o", "tsv"
], capture_output=True, text=True)
FOUNDRY_RESOURCE_ID = result.stdout.strip()

print("3️⃣ Azure AI Project Manager ロール 割り当て (AI Search → Foundry)...")
!az role assignment create \
    --role "Azure AI Project Manager" \
    --assignee $SEARCH_PRINCIPAL_ID \
    --scope $FOUNDRY_RESOURCE_ID

print("\n✅ Foundry IAM 権限 設定 完了!")

## サンプル データ アップロード

Microsoftで 提供するは サンプル データを ダウンロードして Storage Containerに アップロードします.

In [ ]:
# サンプル データ ダウンロード および アップロード
import os
import urllib.request

# サンプル データ URL (Azure Search Sample Data)
sample_files = [
    ("Benefit_Options.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/Benefit_Options.pdf"),
    ("employee_handbook.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/employee_handbook.pdf"),
    ("Northwind_Health_Plus_Benefits_Details.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/Northwind_Health_Plus_Benefits_Details.pdf"),
    ("Northwind_Standard_Benefits_Details.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/Northwind_Standard_Benefits_Details.pdf"),
    ("PerksPlus.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/PerksPlus.pdf"),
    ("role_library.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/role_library.pdf"),
]

# 一時 ディレクトリ 作成
os.makedirs("temp_data", exist_ok=True)

# ファイル ダウンロード
print("📥 サンプル データ ダウンロード 中...")
for filename, url in sample_files:
    filepath = f"temp_data/{filename}"
    if not os.path.exists(filepath):
        print(f"  ダウンロード: {filename}")
        urllib.request.urlretrieve(url, filepath)
    else:
        print(f"  が未 存在: {filename}")

print("\n✅ サンプル データ ダウンロード 完了!")

### Storageに サンプル データ アップロード

In [ ]:
# Storage Containerに ファイル アップロード
print("📤 Storage Containerに アップロード 中...")

for filename, _ in sample_files:
    filepath = f"temp_data/{filename}"
    print(f"  アップロード: {filename}")
    !az storage blob upload \
        --account-name $STORAGE_NAME \
        --container-name documents \
        --file $filepath \
        --name $filename \
        --auth-mode login \
        --overwrite

# アップロードされた ファイル 確認
print("\n📋 アップロードされた ファイル リスト:")
!az storage blob list \
    --account-name $STORAGE_NAME \
    --container-name documents \
    --auth-mode login \
    --query "[].name" \
    --output table

print("\n✅ サンプル データ アップロード 完了!")

# 一時 ファイル 整理 (選択事項)
# import shutil
# shutil.rmtree("temp_data")

## AI Search Index 作成 (Portalで 進行)

**⚠️ 重要**: 次のステップは Azure Portalで 手動で 進行する必要があり します.

### Import Data Wizard 使用 方法:

1. **Azure Portalで AI Search リソース 開く**
   - https://portal.azure.com
   - 作成した AI Search サービス 選択

2. **Import data (new) クリック**
   
3. **Data Source 設定:**
   - Data Source: **Azure Blob Storage**
   - Scenario: **RAG (Retrieval Augmented Generation)**
   - Storage account: `foundry<your-name>`
   - Container: `documents`

4. **Vectorization 設定:**
   - Kind: **Microsoft Foundry**
   - Foundry project: `proj-default`
   - Model deployment: **text-embedding-3-large**
   - Authentication type: **API key**

5. **Semantic Ranker 有効化:**
   - ☑ Enable semantic ranker
   - Schedule: **Once** (初期 インデキシングだけ)

6. **Review + Create**
   - 設定 確認 後 **Create** クリック
   - インデキシング 完了まで 5-10分 所要

完了 後 下 コードで インデックスを 確認してください.

## Python SDKでAI Search Indexを作成

Azure Search Python SDKを 使用して ベクトル 検索が 可能な インデックスを 作成します.

**2つの がない オプション:**
1. **オプション A (推奨)**: ポータルの Import Data Wizard 使用 - 自動で データ インデキシング
2. **オプション B**: 下 Python コード 使用 - インデックスだけ 作成 (データは 別も アップロード 必要)

### Azure Search SDK インストール および Import

In [ ]:
# 必要な パッケージ インストール
%pip install -q azure-search-documents azure-identity

from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, 
    SearchField, 
    SearchFieldDataType,
    VectorSearch, 
    VectorSearchProfile, 
    HnswAlgorithmConfiguration,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch
)
from azure.identity import DefaultAzureCredential

print("✅ パッケージ import 完了")

### AI Search API Key 獲得

In [ ]:
# API キー 方式で 認証 (RBAC 大中)
# サブスクリプション 所有者は API キーを 取得する 数 あります
result = subprocess.run([
    "az", "search", "admin-key", "show",
    "--resource-group", RESOURCE_GROUP,
    "--service-name", SEARCH_NAME,
    "--query", "primaryKey", "-o", "tsv"
], capture_output=True, text=True)

SEARCH_API_KEY = result.stdout.strip()

if SEARCH_API_KEY:
    print("✅ AI Search API キー 獲得 完了")
    print("💡 API キー 方式を 使用すると RBAC ロール ないがも Indexを 作成する 数 あります.")
else:
    print("⚠️ API キー 獲得 失敗. DefaultAzureCredential 方式を 使用します.")

### Search Index 作成 (ベクトル 検索 + Semantic Search)

In [ ]:
# AI Search Index 作成
from azure.core.credentials import AzureKeyCredential

# 認証 方式 選択: API キー まず, なければ DefaultAzureCredential
if 'SEARCH_API_KEY' in globals() and SEARCH_API_KEY:
    credential = AzureKeyCredential(SEARCH_API_KEY)
    auth_method = "API キー"
else:
    credential = DefaultAzureCredential()
    auth_method = "DefaultAzureCredential (RBAC)"

search_endpoint = f"https://{SEARCH_NAME}.search.windows.net"

# SearchIndexClient 作成
index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=credential
)

print(f"📌 AI Search Endpoint: {search_endpoint}")
print(f"📌 作成する Index 名前: {SEARCH_INDEX_NAME}")
print(f"🔐 認証 方式: {auth_method}")

# Index フィールド 定の
fields = [
    # ユニーク 識別子
    SearchField(
        name="chunk_id",
        type=SearchFieldDataType.String,
        key=True,
        sortable=True,
        filterable=True
    ),
    # 親 ドキュメント ID
    SearchField(
        name="parent_id",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    # ドキュメント タイトル
    SearchField(
        name="title",
        type=SearchFieldDataType.String,
        searchable=True,
        filterable=True,
        sortable=True
    ),
    # チャンクされた テキスト 内容
    SearchField(
        name="chunk",
        type=SearchFieldDataType.String,
        searchable=True,
        analyzer_name="ko.microsoft"  # 韓国語 分析期
    ),
    # ベクトル エンベディング (text-embedding-3-large: 3072 次元)
    SearchField(
        name="text_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=3072,
        vector_search_profile_name="my-vector-profile"
    ),
    # メタデータ
    SearchField(
        name="category",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True
    ),
    SearchField(
        name="sourcepage",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    SearchField(
        name="sourcefile",
        type=SearchFieldDataType.String,
        filterable=True
    )
]

# ベクトル 検索 構成
vector_search = VectorSearch(
    profiles=[
        VectorSearchProfile(
            name="my-vector-profile",
            algorithm_configuration_name="my-hnsw-config"
        )
    ],
    algorithms=[
        HnswAlgorithmConfiguration(
            name="my-hnsw-config",
            parameters={
                "m": 4,  # グラフ 接続 数
                "efConstruction": 400,  # インデキシング 時 探索 範上
                "efSearch": 500,  # 検索 時 探索 範上
                "metric": "cosine"  # 類似も 測定 方式
            }
        )
    ]
)

# Semantic Search 構成 (選択 事項)
semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        content_fields=[
            SemanticField(field_name="chunk")
        ]
    )
)

semantic_search = SemanticSearch(
    configurations=[semantic_config]
)

# Index 作成
index = SearchIndex(
    name=SEARCH_INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

try:
    # Index 作成 または 更新
    result = index_client.create_or_update_index(index)
    print(f"\n✅ AI Search Index 作成 完了!")
    print(f"   Index 名前: {result.name}")
    print(f"   フィールド 数: {len(result.fields)}")
    print(f"   ベクトル 検索: 有効化 (3072 次元)")
    print(f"   Semantic Search: 有効化")
    
    # 設定ファイルに 保存
    config["SEARCH_INDEX_NAME"] = SEARCH_INDEX_NAME
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=2)
    
except Exception as e:
    print(f"⚠️ Index 作成 失敗: {e}")
    print("\n💡 可能な 原因:")
    print("   1. AI Search リソースが まだ 準備ならな ない (いくつかの 分 待機)")
    print("   2. 権限 問題:")
    print("      - RBAC 方式: 'Search Index Data Contributor' ロール 必要")
    print("      - サブスクリプション 所有者も Data Plane 権限は 別もで 必要です")
    print("   3. 上の API キー セルを まず 実行すると RBAC ないが が能します")

### 作成された Index 情報 確認

In [ ]:
# (選択 事項) 作成された Index 確認
try:
    # 作成された Index が取得
    created_index = index_client.get_index(SEARCH_INDEX_NAME)
    
    print("📋 Index 詳細 情報:")
    print(f"\nフィールド リスト:")
    for field in created_index.fields:
        field_info = f"  - {field.name} ({field.type})"
        if field.key:
            field_info += " [KEY]"
        if field.searchable:
            field_info += " [検索 が能]"
        if hasattr(field, 'vector_search_dimensions') and field.vector_search_dimensions:
            field_info += f" [ベクトル: {field.vector_search_dimensions}次元]"
        print(field_info)
    
    print(f"\nベクトル 検索 プで必: {len(created_index.vector_search.profiles) if created_index.vector_search else 0}個")
    print(f"Semantic 構成: {len(created_index.semantic_search.configurations) if created_index.semantic_search else 0}個")
    
    # Index リスト 確認
    print("\n📋 全体 Index リスト:")
    indexes = index_client.list_indexes()
    for idx in indexes:
        print(f"  - {idx.name}")
    
except Exception as e:
    print(f"⚠️ Index 情報 取得 失敗: {e}")

## ドキュメント インデキシング (直接 プッシュ 方式)

Storageの PDF ドキュメントを 読んで 直接 Indexに アップロードします.
- PDF パージング および テキスト 抽出
- テキスト チャンキング (2000者 単上)
- Foundry APIで エンベディング 作成
- AI Search Indexに ドキュメント アップロード

### 必須 パッケージ インストール

In [ ]:
# 必要な パッケージ インストール
%pip install -q pypdf azure-storage-blob openai

from azure.storage.blob import BlobServiceClient
from azure.search.documents import SearchClient
from pypdf import PdfReader
from openai import AzureOpenAI
import io
import hashlib
import base64

print("✅ 必要な パッケージ import 完了")

### Storageで PDF ダウンロード および パージング

In [ ]:
# Storageで PDF ファイル ダウンロード
from azure.identity import DefaultAzureCredential

# Blob Service Client 作成
storage_credential = DefaultAzureCredential()
blob_service_client = BlobServiceClient(
    account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
    credential=storage_credential
)

container_client = blob_service_client.get_container_client("documents")

# コンテがあなたの すべての PDF ファイル 私開
print("📥 Storageで PDF ファイル ダウンロード 中...\n")
pdf_files = []

for blob in container_client.list_blobs():
    if blob.name.endswith('.pdf'):
        print(f"  ダウンロード: {blob.name}")
        blob_client = container_client.get_blob_client(blob.name)
        pdf_data = blob_client.download_blob().readall()
        pdf_files.append({
            'name': blob.name,
            'data': pdf_data
        })

print(f"\n✅ 合計 {len(pdf_files)}個 ファイル ダウンロード 完了")

### PDF テキスト チャンキング および メタデータ 作成

In [ ]:
# PDF パージング および テキスト チャンキング
def chunk_text(text, chunk_size=2000, overlap=200):
    """テキストを 指定された サイズで チャンキング"""
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap  # オーバーラップ 適用
    
    return chunks

print("📄 PDF パージング および テキスト チャンキング 中...\n")
all_chunks = []

for pdf_file in pdf_files:
    try:
        # PDF 読み取り
        pdf_reader = PdfReader(io.BytesIO(pdf_file['data']))
        
        # 全体 テキスト 抽出
        full_text = ""
        for page_num, page in enumerate(pdf_reader.pages, 1):
            text = page.extract_text()
            if text:
                full_text += text + "\n"
        
        # テキスト チャンキング
        chunks = chunk_text(full_text)
        
        # メタデータと 一緒に 保存
        for chunk_idx, chunk_content in enumerate(chunks):
            chunk_id = hashlib.md5(f"{pdf_file['name']}_{chunk_idx}".encode()).hexdigest()
            
            all_chunks.append({
                'chunk_id': chunk_id,
                'parent_id': pdf_file['name'],
                'title': pdf_file['name'].replace('.pdf', ''),
                'chunk': chunk_content.strip(),
                'sourcefile': pdf_file['name'],
                'sourcepage': f"page_{chunk_idx + 1}",
                'category': 'health-plan'
            })
        
        print(f"  ✓ {pdf_file['name']}: {len(chunks)}個 チャンク 作成")
        
    except Exception as e:
        print(f"  ⚠️ {pdf_file['name']} パージング 失敗: {e}")

print(f"\n✅ 合計 {len(all_chunks)}個 チャンク 作成 完了")

### Azure OpenAIで エンベディング 作成 および Indexに アップロード

In [ ]:
# Azure OpenAI REST APIで エンベディング 作成
print("🔄 エンベディング 作成 中...\n")

from azure.identity import DefaultAzureCredential
import requests
import json
import time
import subprocess

# エンベディング モデル 設定
embedding_model = "text-embedding-3-large"
api_version = "2024-02-01"

# Azure OpenAI エンドポイント 検索
print("🔍 リソースグループで Azure OpenAI アカウント 見つは 中...")

# Azure CLIで リソースグループの Cognitive Services アカウント 検索
result = subprocess.run([
    "az", "cognitiveservices", "account", "list",
    "--resource-group", RESOURCE_GROUP,
    "--query", "[?kind=='OpenAI'].{name:name, endpoint:properties.endpoint}",
    "-o", "json"
], capture_output=True, text=True)

openai_accounts = json.loads(result.stdout) if result.stdout else []
print(f"📋 発見された OpenAI アカウント: {len(openai_accounts)}個")

if openai_accounts:
    # 最初の 番目 Azure OpenAI アカウント 使用
    openai_account = openai_accounts[0]
    openai_name = openai_account['name']
    openai_endpoint = openai_account['endpoint']
    
    print(f"✅ Azure OpenAI アカウント: {openai_name}")
    print(f"🔗 Endpoint: {openai_endpoint}")
    
    # デプロイされた モデル 確認
    result = subprocess.run([
        "az", "cognitiveservices", "account", "deployment", "list",
        "--name", openai_name,
        "--resource-group", RESOURCE_GROUP,
        "--query", "[].{name:name, model:properties.model.name}",
        "-o", "json"
    ], capture_output=True, text=True)
    
    deployments = json.loads(result.stdout) if result.stdout else []
    print(f"\n📦 デプロイされた モデル:")
    for dep in deployments:
        print(f"   - {dep['name']}: {dep['model']}")
    
    # embedding モデル 検索
    embedding_deployment = None
    for dep in deployments:
        if "embedding" in dep['name'].lower() or "embedding" in dep['model'].lower():
            embedding_deployment = dep['name']
            print(f"\n✅ エンベディング モデル デプロイ 発見: {embedding_deployment}")
            break
    
    if not embedding_deployment:
        # デフォルト 名前で 時も
        embedding_deployment = embedding_model
        print(f"\n💡 デフォルト デプロイ 名前 使用: {embedding_deployment}")
    
    # REST API URL 構成
    embeddings_url = f"{openai_endpoint.rstrip('/')}/openai/deployments/{embedding_deployment}/embeddings?api-version={api_version}"
    token_scope = "https://cognitiveservices.azure.com/.default"
    
else:
    print("⚠️ Azure OpenAI アカウントを 見つを 数 ありません。")
    print("💡 Foundry リソース 自体を 使用してみます...")
    
    # Foundry リソースの エンドポイント が取得
    result = subprocess.run([
        "az", "cognitiveservices", "account", "show",
        "--name", FOUNDRY_NAME,
        "--resource-group", RESOURCE_GROUP,
        "--query", "properties.endpoint",
        "-o", "tsv"
    ], capture_output=True, text=True)
    
    foundry_endpoint = result.stdout.strip()
    print(f"🔗 Foundry Endpoint: {foundry_endpoint}")
    
    embeddings_url = f"{foundry_endpoint.rstrip('/')}/openai/deployments/{embedding_model}/embeddings?api-version={api_version}"
    token_scope = "https://cognitiveservices.azure.com/.default"

print(f"\n📍 エンベディング API URL: {embeddings_url}")
print(f"🔐 認証 Scope: {token_scope}")

# Azure 認証 トークン が取得
credential = DefaultAzureCredential()

# バッチで エンベディング 作成 (Rate limit 中静的 処理)
batch_size = 30  # Rate limit 考慮して 作は バッチ
embedded_chunks = []
max_retries = 5
base_retry_delay = 30  # デフォルト 待機 時間 30秒

print(f"\n⚡ エンベディング 作成 開始 (合計 {len(all_chunks)}個, バッチ当 {batch_size}個)")
print(f"💡 中静的の 処理を 上して Rate limit 発生 時 待機します.\n")

for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]
    batch_texts = [chunk['chunk'] for chunk in batch]
    batch_num = i // batch_size + 1
    total_batches = (len(all_chunks) + batch_size - 1) // batch_size
    
    success = False
    for retry in range(max_retries):
        try:
            # アクセス トークン が取得
            token = credential.get_token(token_scope)
            
            # REST API 呼び出し
            headers = {
                "Content-Type": "application/json",
                "Authorization": f"Bearer {token.token}"
            }
            
            payload = {
                "input": batch_texts,
                "dimensions": 3072
            }
            
            response = requests.post(
                embeddings_url,
                headers=headers,
                json=payload,
                timeout=60
            )
            
            if response.status_code == 200:
                result = response.json()
                
                # エンベディングを チャンクに 追加
                for j, chunk in enumerate(batch):
                    chunk['text_vector'] = result['data'][j]['embedding']
                    embedded_chunks.append(chunk)
                
                print(f"  ✅ バッチ {batch_num}/{total_batches}: {len(embedded_chunks)}/{len(all_chunks)} チャンク 完了")
                success = True
                break
                
            elif response.status_code == 429:
                # Rate limit エラー - 指数 バックオフで 再試も
                if retry < max_retries - 1:
                    wait_time = base_retry_delay * (2 ** retry)  # 30, 60, 120, 240秒...
                    print(f"  ⏳ バッチ {batch_num}/{total_batches}: Rate limit. {wait_time}秒 待機 後 再試も ({retry+1}/{max_retries})...")
                    time.sleep(wait_time)
                else:
                    print(f"  ⚠️ バッチ {batch_num}/{total_batches}: 最大 再試も 回数 秒と - スキップ")
                    for chunk in batch:
                        chunk['text_vector'] = None
                        embedded_chunks.append(chunk)
                    success = True  # 次 バッチで 進行
                    break
            else:
                print(f"  ⚠️ バッチ {batch_num}/{total_batches} 失敗 (HTTP {response.status_code}): {response.text[:150]}")
                for chunk in batch:
                    chunk['text_vector'] = None
                    embedded_chunks.append(chunk)
                success = True  # 次 バッチで 進行
                break
            
        except Exception as e:
            print(f"  ⚠️ バッチ {batch_num}/{total_batches} エラー: {str(e)[:100]}")
            if retry < max_retries - 1:
                print(f"     再試も {retry + 1}/{max_retries}...")
                time.sleep(5)
            else:
                for chunk in batch:
                    chunk['text_vector'] = None
                    embedded_chunks.append(chunk)
                success = True
                break
    
    # バッチ 間 1秒 待機 (Rate limit 防止)
    if i + batch_size < len(all_chunks) and success:
        time.sleep(1)

print(f"\n✅ エンベディング 作成 完了: {len(embedded_chunks)}個 チャンク")
print(f"📊 ベクトル 次元: {len(embedded_chunks[0]['text_vector']) if embedded_chunks and embedded_chunks[0].get('text_vector') else 0}次元")

### Foundry MIに Search Index Data Reader ロール 割り当て

In [ ]:
# Indexに ドキュメント アップロード
from azure.core.credentials import AzureKeyCredential

# API キーで SearchClient 作成 (RBAC 代わりに API キー 使用)
upload_credential = AzureKeyCredential(SEARCH_API_KEY) if SEARCH_API_KEY else DefaultAzureCredential()

search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=SEARCH_INDEX_NAME,
    credential=upload_credential
)

print("📤 Indexに ドキュメント アップロード 中...")
print(f"🔐 認証 方式: {'API キー' if SEARCH_API_KEY else 'DefaultAzureCredential'}\n")

# バッチ アップロード (最大 1000個ずつ)
batch_size = 30
uploaded_count = 0

for i in range(0, len(embedded_chunks), batch_size):
    batch = embedded_chunks[i:i+batch_size]
    
    # None ベクトル 削除 (エンベディング 失敗した 場合)
    valid_batch = [chunk for chunk in batch if chunk['text_vector'] is not None]
    
    if not valid_batch:
        continue
    
    try:
        # ドキュメント アップロード
        result = search_client.upload_documents(documents=valid_batch)
        
        # 成功した ドキュメント 数 カウント
        succeeded = sum(1 for r in result if r.succeeded)
        uploaded_count += succeeded
        
        print(f"  アップロード: {uploaded_count}/{len([c for c in embedded_chunks if c['text_vector']])} チャンク")
        
    except Exception as e:
        print(f"  ⚠️ バッチ {i//batch_size + 1} アップロード 失敗: {e}")

print(f"\n✅ インデキシング 完了!")
print(f"   合計 アップロード: {uploaded_count}個 ドキュメント")
print(f"   Index: {SEARCH_INDEX_NAME}")
print(f"\n💡 Index 確認:")
print(f"   https://portal.azure.com → {SEARCH_NAME} → Indexes → {SEARCH_INDEX_NAME}")

### Foundryに AI Search Connection 作成 (REST API)

## Knowledge Baseの作成 (コード ベース)

**重要**: Knowledge Base APIは プレビュー 機能で, プレビュー バージョン SDKが 必要です.

### Azure Search Documents プレビュー バージョン インストール

In [ ]:
# Knowledge Base API サポートを 上した プレビュー バージョン インストール
print("📦 プレビュー SDK インストール 中...")
%pip install -q --upgrade azure-search-documents==11.7.0b2

print("\n✅ azure-search-documents 11.7.0b2 インストール 完了!")
print("   (Knowledge Base API サポート)")

print("\n⚠️  重要: カーネル 再開始 必要!")
print("   1. 上部 メニュー: Kernel > Restart Kernel")
print("   2. または 下 セル 実行して 自動 再開始")
print("\n💡 カーネル 再開始 後 次 セルから 続行 進行してください.")

## Foundry Managed Identityに AI Search 権限 付与 (**必須**)

**⚠️ 重要**: API Key Connectionだけでは **Agent 実行が 不が能**します!

**権限 構造:**
- 🔑 **API Key Connection** → ポータルで Knowledge Base 取得用 (Cell 60)
- 🤖 **Managed Identity 権限** → Agentが Knowledge Base 実行用 (**が セクション**)

**なぜ 必要なが?**
- Agent 実行 時 Foundryの Managed Identityで Knowledge Base MCP エンドポイントに アクセス
- API Keyは Connection 設定用がであり, Agent ランタイムには 使用ならな ない
- 403 Forbidden エラー 防止を 上して 必ず 設定 必要

In [ ]:
# Foundry Managed Identityに AI Search 権限 付与 (Agent 実行用 - 必須!)
print("🔐 Foundry Managed Identityに AI Search 権限 設定 中...")
print("=" * 80)

import json
import uuid
import requests
from azure.identity import DefaultAzureCredential
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters

try:
    # config ファイルで 必要な 情報 ロード
    config_file = '.foundry_config.json'
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    SEARCH_NAME = config.get("SEARCH_NAME")
    SUBSCRIPTION_ID = config.get("AZURE_SUBSCRIPTION_ID") or config.get("SUBSCRIPTION_ID")
    
    # Foundry リソース ID
    foundry_account_id = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_NAME}"
    
    # AI Search リソース ID
    search_resource_id = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.Search/searchServices/{SEARCH_NAME}"
    
    credential = DefaultAzureCredential()
    
    # Foundryの Managed Identity Principal ID が取得
    print(f"\n🔍 Foundry Managed Identity 確認 中...")
    
    token = credential.get_token("https://management.azure.com/.default")
    headers = {
        "Authorization": f"Bearer {token.token}",
        "Content-Type": "application/json"
    }
    
    foundry_url = f"https://management.azure.com{foundry_account_id}?api-version=2025-09-01"
    response = requests.get(foundry_url, headers=headers)
    
    if response.status_code != 200:
        raise Exception(f"Foundry 情報 取得 失敗: {response.status_code}")
    
    foundry_data = response.json()
    principal_id = foundry_data.get("identity", {}).get("principalId")
    
    if not principal_id:
        print(f"   ⚠️ Foundryの Managed Identityが 有効化ならな ませんでした.")
        print(f"   💡 Managed Identity 有効化 中...")
        
        # Managed Identity 有効化
        foundry_data["identity"] = {"type": "SystemAssigned"}
        update_response = requests.patch(foundry_url, headers=headers, json={"identity": {"type": "SystemAssigned"}})
        
        if update_response.status_code in [200, 201]:
            principal_id = update_response.json().get("identity", {}).get("principalId")
            print(f"   ✅ Managed Identity 有効化 完了!")
        else:
            raise Exception(f"Managed Identity 有効化 失敗: {update_response.text}")
    
    print(f"   ✅ Principal ID: {principal_id}")
    
    # Role Assignment 作成
    auth_client = AuthorizationManagementClient(credential, SUBSCRIPTION_ID)
    
    # 必要な ロール 定の (Agent 実行用)
    roles_to_assign = [
        {
            "name": "Search Index Data Reader",
            "id": "1407120a-92aa-4202-b7e9-c0e197c71c8f",
            "purpose": "Knowledge Base データ 読み取り"
        },
        {
            "name": "Search Service Contributor",
            "id": "7ca78c08-252a-4471-8644-bb5ff32d4ba0",
            "purpose": "Knowledge Base MCP エンドポイント アクセス"
        }
    ]
    
    for role in roles_to_assign:
        print(f"\n🚀 '{role['name']}' 権限 確認 中...")
        print(f"   目的: {role['purpose']}")
        
        role_definition_id = f"/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.Authorization/roleDefinitions/{role['id']}"
        
        # 既存の role assignment 確認
        existing_assignments = list(auth_client.role_assignments.list_for_scope(
            scope=search_resource_id,
            filter=f"principalId eq '{principal_id}'"
        ))
        
        has_role = any(
            role['id'] in assignment.role_definition_id
            for assignment in existing_assignments
        )
        
        if has_role:
            print(f"   ✅ '{role['name']}' 権限が が未 存在します.")
        else:
            # 新しい Role Assignment 作成
            role_assignment_name = str(uuid.uuid4())
            role_assignment_params = RoleAssignmentCreateParameters(
                role_definition_id=role_definition_id,
                principal_id=principal_id,
                principal_type="ServicePrincipal"
            )
            
            assignment = auth_client.role_assignments.create(
                scope=search_resource_id,
                role_assignment_name=role_assignment_name,
                parameters=role_assignment_params
            )
            
            print(f"   ✅ '{role['name']}' 権限 付与 完了!")
            print(f"   Assignment ID: {assignment.name}")
    
    print(f"\n" + "=" * 80)
    print(f"✅ Agent 実行用 権限 設定 完了!")
    print(f"\n⚠️ 重要: 権限が 伝播されはところ 2-3分 所要なります.")
    print(f"💡 2-3分 後:")
    print(f"   1. Agent テスト 実行 が能")
    print(f"   2. 403 Forbidden エラー 解決")
    print(f"   3. Knowledge Base MCP エンドポイント 正常 アクセス")
    
except Exception as e:
    import traceback
    print(f"\n⚠️ 権限 設定 失敗: {e}")
    print("\n詳細 エラー:")
    traceback.print_exc()


### Azure AI Search 接続 追加

Foundry IQで Knowledge Baseを 使用するには まず AI Search リソースを 接続する必要があり します.

**Foundry IQで AI Search 接続すること:**

1. **Azure AI Foundry Portal 接続**
   - https://ai.azure.com 接続
   - 作成した Foundry プロジェクト 選択

2. **Foundry IQ メニューで が同**
   - 左側 メニューで **Foundry IQ** クリック
   - "Ground your agent in enterprise knowledge" 画面が 表示なる

3. **AI Search リソース 接続**
   - **Azure AI Search resource** ドロップダウン クリック
   - 作成した AI Search サービス 選択
   - **Connect** ボタン クリック

4. **接続 完了**
   - AI Search リソースが 成功的で 接続なると Knowledge basesと Indexes タブが 有効化なります

> **💡 Tip**: 
> - AI Search リソースが リストに なければ "Create new resource" リンクを クリックして 新しいで 作成する 数 あります
> - 接続 後には コードで `project_client.connections.list()`で 接続を 確認する 数 あります

## Knowledge Baseの作成

In [ ]:
# Knowledge Baseを 上した ライブラリ import
import json
import subprocess
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    KnowledgeBaseAzureOpenAIModel,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    SearchIndexFieldReference,
    AzureOpenAIVectorizerParameters,
    KnowledgeRetrievalOutputMode
)

# config ファイル ロード
config_file = '.foundry_config.json'
with open(config_file, 'r') as f:
    config = json.load(f)

SEARCH_NAME = config.get("SEARCH_NAME")

# AI Search API キー 獲得 (コマンドで 取得)
result = subprocess.run([
    "az", "search", "admin-key", "show",
    "--resource-group", RESOURCE_GROUP,
    "--service-name", SEARCH_NAME,
    "--query", "primaryKey", "-o", "tsv"
], capture_output=True, text=True)

SEARCH_API_KEY = result.stdout.strip()
search_endpoint = f"https://{SEARCH_NAME}.search.windows.net"

# SearchIndexClient 初期化 (API Key 使用)
index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=AzureKeyCredential(SEARCH_API_KEY)
)

print("✅ Knowledge Base ライブラリ import 完了")
print(f"   - KnowledgeBase")
print(f"   - KnowledgeSourceReference")
print(f"   - KnowledgeBaseAzureOpenAIModel")
print(f"   - SearchIndexKnowledgeSource")
print(f"   - SearchIndexKnowledgeSourceParameters")
print(f"   - SearchIndexFieldReference")
print(f"✅ SearchIndexClient 初期化 完了: {search_endpoint}")


### Knowledge Source 作成 (Search Index 接続)

In [ ]:
print("🔗 1ステップ: Knowledge Source 作成 中...")
KNOWLEDGE_SOURCE_NAME = "knowledge-source-01"

try:
    knowledge_source = SearchIndexKnowledgeSource(
        name=KNOWLEDGE_SOURCE_NAME,
        description="Foundry IQ Knowledge Source from existing search index",
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=SEARCH_INDEX_NAME,
            # 検索に 使用する フィールドたち 指定 (実際 インデックス フィールド 名前 使用)
            source_data_fields=[
                SearchIndexFieldReference(name="chunk"),  # テキスト コンテンツ
                SearchIndexFieldReference(name="title")   # ドキュメント タイトル
            ],
            # 検索 フィールド (キー フィールド - 実際 インデックスの キー フィールド 名前 使用)
            search_fields=[
                SearchIndexFieldReference(name="chunk_id")
            ]
        )
    )
    
    result_ks = index_client.create_or_update_knowledge_source(knowledge_source)
    print(f"   ✅ Knowledge Source 作成 完了: {KNOWLEDGE_SOURCE_NAME}")
except Exception as e:
    print(f"   ⚠️ Knowledge Source 作成 失敗: {e}")
    print("   💡 が未 存在するは 場合 無視して 続行 進行します.")

### Knowledge Baseの作成

In [ ]:
print("📚 2ステップ: Knowledge Base 作成 中...")
KNOWLEDGE_BASE_NAME = "foundry-knowledge-base"

# configで 必要な 変数 ロード
FOUNDRY_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
SEARCH_INDEX_NAME = config.get("SEARCH_INDEX_NAME", "knowledge-index")
KNOWLEDGE_SOURCE_NAME = config.get("KNOWLEDGE_SOURCE_NAME", "knowledge-source-01")

# Azure OpenAI パラメータ 設定
# answer synthesisを 上した GPT モデル 使用
aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=FOUNDRY_ENDPOINT,  # Foundry エンドポイント 使用
    deployment_name="gpt-4.1",  # GPT デプロイ 名前
    model_name="gpt-4.1"
)

try:
    knowledge_base = KnowledgeBase(
        name=KNOWLEDGE_BASE_NAME,
        description="Foundry IQ Knowledge Base for HR documents",
        retrieval_instructions="が ナレッジベースは HR 関連 ドキュメントを 含むして あります. 従業員 福利厚生, 採用 ポリシー などに に対する 質問に 回答してください.",
        answer_instructions="検索された ドキュメントを ベースで 名確で 間決した 回答を 提供してください. ドキュメントで 直接 引用して 信頼性を 高がください。",
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,  # 自動 回答 合成
        knowledge_sources=[
            KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)
        ],
        models=[
            KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)
        ]
    )
    
    result_kb = index_client.create_or_update_knowledge_base(knowledge_base)
    print(f"   ✅ Knowledge Base 作成 完了!")
    print(f"\n📋 Knowledge Base 情報:")
    print(f"   名前: {KNOWLEDGE_BASE_NAME}")
    print(f"   Knowledge Source: {KNOWLEDGE_SOURCE_NAME}")
    print(f"   Search Index: {SEARCH_INDEX_NAME}")
    print(f"   Output Mode: Answer Synthesis")
    
    # configに 保存
    config["KNOWLEDGE_BASE_NAME"] = KNOWLEDGE_BASE_NAME
    config["KNOWLEDGE_SOURCE_NAME"] = KNOWLEDGE_SOURCE_NAME
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"\n💡 Knowledge Base 確認:")
    print(f"   https://portal.azure.com → {SEARCH_NAME} → Knowledge bases")
    
except Exception as e:
    import traceback
    print(f"   ⚠️ Knowledge Base 作成 失敗: {e}")
    print("\n詳細 エラー:")
    traceback.print_exc()
    print("\n💡 問題 解決:")
    print("   1. カーネルを 再開始したはない 確認")
    print("   2. azure-search-documents プレビュー バージョン(11.7.0b2)が インストールなったはない 確認")
    print("   3. Azure OpenAI モデル 'gpt-4.1'が デプロイなって あるはない 確認")
    print("   4. Search API Key 権限が 充分したか 確認")

### KnowledgeAgentの作成

In [ ]:
# Connectionに API Key 設定 (403 Forbidden 解決)
print("🔑 AI Search Connection API Key 設定 中...")
print("=" * 80)

import json
import subprocess
import requests
from azure.identity import DefaultAzureCredential

try:
    # config ファイルで 必要な 情報 ロード
    config_file = '.foundry_config.json'
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    SEARCH_NAME = config.get("SEARCH_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    SUBSCRIPTION_ID = config.get("AZURE_SUBSCRIPTION_ID") or config.get("SUBSCRIPTION_ID")
    PROJECT_ENDPOINT = config.get("PROJECT_ENDPOINT") or config.get("FOUNDRY_ENDPOINT")
    
    # AI Search API キー 獲得
    print(f"\n🔍 AI Search API Key 取得 中...")
    result = subprocess.run([
        "az", "search", "admin-key", "show",
        "--resource-group", RESOURCE_GROUP,
        "--service-name", SEARCH_NAME,
        "--query", "primaryKey", "-o", "tsv"
    ], capture_output=True, text=True)
    
    if result.returncode != 0:
        raise Exception(f"API Key 取得 失敗: {result.stderr}")
    
    api_key = result.stdout.strip()
    print(f"   ✅ API Key 獲得 完了")
    
    # Project Connection ID 確認
    from azure.ai.projects import AIProjectClient
    credential = DefaultAzureCredential()
    project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
    
    connections = project_client.connections.list()
    search_connections = [c for c in connections if 'search' in c.name.lower()]
    
    if not search_connections:
        raise ValueError("AI Search Connectionを 見つを 数 ありません。")
    
    connection = search_connections[0]
    CONNECTION_NAME = connection.name
    
    print(f"\n🔧 Connection 更新 中: {CONNECTION_NAME}")
    
    # Connection 更新 (REST API 使用)
    token = credential.get_token("https://management.azure.com/.default")
    headers = {
        "Authorization": f"Bearer {token.token}",
        "Content-Type": "application/json"
    }
    
    # Project リソース ID 抽出
    project_id = PROJECT_ENDPOINT.split("/projects/")[0].replace("https://", "").replace(".api.azureml.ms", "")
    project_name = PROJECT_ENDPOINT.split("/projects/")[1]
    
    # Foundry アカウント 情報 確認
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    foundry_base_url = f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_NAME}"
    
    # Connection URL
    connection_url = f"{foundry_base_url}/projects/{project_name}/connections/{CONNECTION_NAME}?api-version=2025-09-01"
    
    # Connection 情報 取得
    response = requests.get(connection_url, headers=headers)
    if response.status_code == 200:
        connection_data = response.json()
        
        # API Keyで 更新
        connection_data["properties"]["credentials"] = {
            "type": "ApiKey",
            "key": api_key
        }
        
        # Connection 更新
        update_response = requests.put(connection_url, headers=headers, json=connection_data)
        
        if update_response.status_code in [200, 201]:
            print(f"   ✅ Connection API Key 設定 完了!")
            print(f"\n💡 が第 Agentが Knowledge Baseに アクセスする 数 あります.")
        else:
            print(f"   ⚠️ Connection 更新 失敗: {update_response.status_code}")
            print(f"   レスポンス: {update_response.text}")
    else:
        print(f"   ⚠️ Connection 取得 失敗: {response.status_code}")
        print(f"   レスポンス: {response.text}")
        
        # 大中: Portalで 手動 設定 中内
        print(f"\n📝 手動 設定 方法:")
        print(f"   1. https://ai.azure.com 接続")
        print(f"   2. Build → Connections → {CONNECTION_NAME} 選択")
        print(f"   3. 'Edit' クリック")
        print(f"   4. Authentication セクションで 'API Key' 選択")
        print(f"   5. API Key 入力 後 保存")
    
    print(f"\n" + "=" * 80)
    print(f"✅ Connection 設定 完了!")
    
except Exception as e:
    import traceback
    print(f"\n⚠️ Connection 設定 失敗: {e}")
    print("\n詳細 エラー:")
    traceback.print_exc()
    print(f"\n💡 手動 設定:")
    print(f"   Portalで Connectionの API Keyを 直接 設定してください.")


### Connection 作成 (API Key 認証)

Agentが Knowledge Baseに アクセスするには AI Search Connectionが API Keyで 認証ならなければ します.

In [ ]:
print("🤖 3ステップ: KnowledgeAgent 作成 中...")
print("=" * 80)

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from azure.identity import DefaultAzureCredential

try:
    # config ファイルで 必要な 情報 ロード
    config_file = '.foundry_config.json'
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    PROJECT_ENDPOINT = config.get("PROJECT_ENDPOINT") or config.get("FOUNDRY_ENDPOINT")
    SEARCH_NAME = config["SEARCH_NAME"]
    KNOWLEDGE_BASE_NAME = config.get("KNOWLEDGE_BASE_NAME", "foundry-knowledge-base")
    
    # AIProjectClient 作成 (Connection 取得用)
    credential = DefaultAzureCredential()
    project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
    
    # Connection 名前 確認 (実際 存在するは Connection 使用)
    print(f"\n🔍 プロジェクト Connection リスト 確認 中...")
    connections = project_client.connections.list()
    search_connections = [c for c in connections if 'search' in c.name.lower()]
    
    if not search_connections:
        raise ValueError(f"❌ AI Search Connectionを 見つを 数 ありません。\n"
                        f"💡 Portalで AI Search Connectionを まず 作成してください:\n"
                        f"   https://ai.azure.com → Build → Connections → + Connection → Azure AI Search")
    
    # 最初の 番目 Search Connection 使用
    PROJECT_CONNECTION_NAME = search_connections[0].name
    print(f"   ✅ Connection 発見: {PROJECT_CONNECTION_NAME}")
    
    # MCP エンドポイント URL (api-version 含む, API Keyは Connectionを を通じて 伝達)
    MCP_ENDPOINT = f"https://{SEARCH_NAME}.search.windows.net/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version=2025-11-01-preview"
    
    print(f"\n📋 Agent 設定:")
    print(f"   Project Endpoint: {PROJECT_ENDPOINT}")
    print(f"   Connection: {PROJECT_CONNECTION_NAME}")
    print(f"   MCP Endpoint: {MCP_ENDPOINT}")
    print(f"\n💡 認証 方式: Connectionを を通じた API Key 自動 伝達")
    
    # Agent Instructions (Microsoft 推奨 テンプレート)
    KNOWLEDGE_INSTRUCTIONS = """You are a helpful assistant that must use the knowledge base to answer all the questions from user. You must never answer from your own knowledge under any circumstances.

Every answer must always provide annotations for using the MCP knowledge base tool and render them as: 【message_idx:search_idx†source_name】

If you cannot find the answer in the provided knowledge base you must respond with "I don't know".

韓国語で 回答してください."""
    
    # MCP Tool 設定 (Connectionを を通じて API Key 伝達)
    print(f"\n🔧 MCP Tool 構成 中...")
    print(f"   Server Label: knowledge-base")
    print(f"   Connection ID: {PROJECT_CONNECTION_NAME} (API Key 認証)")
    
    mcp_kb_tool = MCPTool(
        server_label="knowledge-base",
        server_url=MCP_ENDPOINT,
        require_approval="never",
        allowed_tools=["knowledge_base_retrieve"],
        project_connection_id=PROJECT_CONNECTION_NAME
    )
    
    print(f"\n🚀 Agent 作成 中...")
    
    # Agent 作成
    agent = project_client.agents.create_version(
        agent_name="KnowledgeAgent",
        definition=PromptAgentDefinition(
            model="gpt-5.1",
            instructions=KNOWLEDGE_INSTRUCTIONS,
            tools=[mcp_kb_tool]
        )
    )
    
    print(f"   ✅ KnowledgeAgent 作成 完了!")
    print(f"\n📋 Agent 情報:")
    print(f"   名前: {agent.name}")
    print(f"   バージョン: {agent.version}")
    print(f"   モデル: gpt-5.1")
    print(f"   MCP Tool: knowledge_base_retrieve")
    
    # configに 保存
    config["AGENT_NAME"] = agent.name
    config["AGENT_VERSION"] = agent.version
    config["PROJECT_CONNECTION_NAME"] = PROJECT_CONNECTION_NAME  # 実際 Connection 名前 保存
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"\n💡 Agent 確認:")
    print(f"   https://ai.azure.com → Build → Agents → {agent.name}")
    
except Exception as e:
    import traceback
    print(f"   ⚠️ Agent 作成 失敗: {e}")
    print("\n詳細 エラー:")
    traceback.print_exc()
    print("\n💡 問題 解決:")
    print("   1. Project Connectionが まず 作成なったはない 確認")
    print("   2. Managed Identity 権限が 設定なったはない 確認")
    print("   3. Azure CLI ログの 状態 確認")
    print("   4. gpt-5.1 モデルが デプロイなって あるはない 確認")

### Agent テスト

> **⚠️ 実行 前 必須 作業**: [Azure AI Foundry Portal](https://ai.azure.com)で `foundry-knowledge-base` Knowledge Baseを Projectに 接続する必要があり します.

In [ ]:
# KnowledgeAgent テスト
print("🧪 4ステップ: KnowledgeAgent テスト 開始...")
print("=" * 80)

try:
    # Agent 変数 検証
    if 'agent' in locals():
        knowledge_agent = agent
        print(f"✅ Agent 変数 使用: {knowledge_agent.name} (version: {knowledge_agent.version})")
    elif 'knowledge_agent' not in locals():
        raise NameError("knowledge_agentが 定のならな ませんでした. 上の KnowledgeAgent 作成 セルを まず 実行してください.")
    
    # OpenAI Client 検証
    if 'openai_client' not in locals():
        raise NameError("OpenAI clientが ありません。 Setup セルを まず 実行してください.")
    
    # Test Questions (最初の 番目だけ テスト)
    test_questions = [
        "PerkPlusが カバーするは 項目たちを 教えて"
    ]
    
    for i, question in enumerate(test_questions, 1):
        print(f"\n[質問 {i}]: {question}")
        print("-" * 80)
        
        # Conversation 作成
        conversation = openai_client.conversations.create()
        
        # Agentを を通じて レスポンス 作成
        print(f"  レスポンス 作成 中...", flush=True)
        response = openai_client.responses.create(
            conversation=conversation.id,
            input=question,
            extra_body={"agent": {"name": knowledge_agent.name, "type": "agent_reference"}}
        )
        
        # DEBUG: Response 状態 確認
        print(f"\n🔍 Response 状態: {response.status}")
        print(f"🔍 Response ID: {response.id}")
        
        if hasattr(response, 'output') and isinstance(response.output, list):
            print(f"🔍 Output 長さが: {len(response.output)}")
            for idx, item in enumerate(response.output):
                print(f"\n  Item {idx}: {item.__class__.__name__}")
                if hasattr(item, 'type'):
                    print(f"    type: {item.type}")
                if hasattr(item, 'id'):
                    print(f"    id: {item.id}")
                if hasattr(item, 'text'):
                    print(f"    text (最初 100者): {item.text[:100]}")
                if item.__class__.__name__ == 'McpApprovalRequest':
                    print(f"    ⚠️ Approval 必要!")
                    print(f"    name: {item.name if hasattr(item, 'name') else 'N/A'}")
                    print(f"    arguments: {item.arguments[:200] if hasattr(item, 'arguments') else 'N/A'}")
        
        # レスポンス テキスト 抽出
        actual_text = ""
        
        if hasattr(response, 'output') and isinstance(response.output, list):
            for item in response.output:
                if hasattr(item, 'text'):
                    actual_text = item.text
                    break
        
        print(f"\n[レスポンス]: {actual_text if actual_text else '⚠️ レスポンスを 見つを 数 ありません'}\n")
    
    print("=" * 80)
    print("✅ KnowledgeAgent テスト 完了!")
    
except Exception as e:
    import traceback
    print(f"\n⚠️ テスト 失敗: {e}")
    traceback.print_exc()

## Azure Blob Storage ベース Knowledge Base

Blob Storageを 直接 接続して より 間簡単に Knowledge Baseを 作成します.

### 1. Storage Account 情報 確認

In [ ]:
# config ファイルで Storage Account 名前 ロード
try:
    STORAGE_NAME = config.get("STORAGE_NAME")
    if not STORAGE_NAME:
        raise ValueError("STORAGE_NAMEが configに ありません。")
except:
    print("⚠️ configで STORAGE_NAMEを 見つを 数 ありません。")
    print("💡 上の '環境変数 ロード' セルと 'AI Search および Storage リソース 名前 作成' セルを まず 実行してください.")
    raise

# Container 名前 設定
CONTAINER_NAME = "documents"

print(f"Storage Account: {STORAGE_NAME}")
print(f"Container: {CONTAINER_NAME}")
print(f"\n✅ が未 作成された Storage Accountと Containerを 使用します.")
print(f"💡 同じ Blob Storageを 使用するためで 追加 設定 不要")

### 2. Knowledge Source 作成 (Blob Storage)

In [ ]:
import subprocess
from azure.search.documents.indexes.models import (
    AzureBlobKnowledgeSource,
    AzureBlobKnowledgeSourceParameters,
    KnowledgeSourceIngestionParameters,
    KnowledgeSourceContentExtractionMode,
    KnowledgeSourceAzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters
)

# Knowledge Source 名前
BLOB_KNOWLEDGE_SOURCE_NAME = "ks-azureblob-200"

print("Knowledge Source 作成 中...")
print(f"名前: {BLOB_KNOWLEDGE_SOURCE_NAME}")

try:
    # Storage Account Resource ID 構成
    storage_resource_id = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.Storage/storageAccounts/{STORAGE_NAME}"
    
    # Embedding Model (Vectorizer) 設定
    aoai_vectorizer_params = AzureOpenAIVectorizerParameters(
        resource_url=FOUNDRY_ENDPOINT,
        deployment_name="text-embedding-3-large",
        model_name="text-embedding-3-large"
    )
    
    embedding_vectorizer = KnowledgeSourceAzureOpenAIVectorizer(
        azure_open_ai_parameters=aoai_vectorizer_params
    )
    
    # Ingestion パラメータ 設定 (System-Assigned MIは identity パラメータ 不要)
    ingestion_params = KnowledgeSourceIngestionParameters(
        content_extraction_mode=KnowledgeSourceContentExtractionMode.MINIMAL,
        embedding_model=embedding_vectorizer
    )
    
    # Blob Storage パラメータ 設定 (Managed Identity Connection String 使用)
    # System-Assigned Managed Identityを 使用する 時は ResourceId 形式の connection stringだけ 必要
    managed_identity_connection_string = f"ResourceId={storage_resource_id};"
    
    blob_params = AzureBlobKnowledgeSourceParameters(
        connection_string=managed_identity_connection_string,
        container_name=CONTAINER_NAME,
        ingestion_parameters=ingestion_params
    )
    
    # Blob Storage ベース Knowledge Source 作成
    blob_knowledge_source = AzureBlobKnowledgeSource(
        name=BLOB_KNOWLEDGE_SOURCE_NAME,
        azure_blob_parameters=blob_params,
        description="Blob Storage ベース Knowledge Source"
    )
    
    # Knowledge Source 作成
    result_blob_ks = index_client.create_knowledge_source(blob_knowledge_source)
    
    print(f"\n✅ Knowledge Source 作成 完了!")
    print(f"   - 名前: {result_blob_ks.name}")
    print(f"   - タイプ: Blob Storage")
    print(f"   - Container: {CONTAINER_NAME}")
    print(f"   - Embedding モデル: text-embedding-3-large")
    print(f"\n💡 が Knowledge Sourceは Blob Storageの ファイルを 自動で インデキシングします.")
    
except Exception as e:
    import traceback
    print(f"\n⚠️ Knowledge Source 作成 失敗: {e}")
    print("\n詳細 エラー:")
    traceback.print_exc()
    print("\n💡 解決 方法:")
    print("   1. Storage Accountと Containerが 正しいか 確認")
    print("   2. AI Search Managed Identityが Storageに に対する 権限が あるはない 確認")
    print("   3. FOUNDRY_ENDPOINTが 正しく 設定なったはない 確認")
    print("   4. が未 存在するは 場合 異なる 名前 使用")

### 3. Knowledge Base 作成 (Blob Storage ベース)

> **⚠️ 実行 前 必須 作業**: [Azure AI Foundry Portal](https://ai.azure.com)で `blob-knowledge-base` Knowledge Baseを Projectに 接続する必要があり します.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeSourceReference,
    KnowledgeBaseAzureOpenAIModel
)

# Knowledge Base 名前
BLOB_KNOWLEDGE_BASE_NAME = "knowledgebase200"

print("Knowledge Base 作成 中...")
print(f"名前: {BLOB_KNOWLEDGE_BASE_NAME}")

try:
    # Azure OpenAI モデル 設定 (Answer Synthesis用)
    aoai_model_params = AzureOpenAIVectorizerParameters(
        resource_url=FOUNDRY_ENDPOINT,
        deployment_name="gpt-5.1",
        model_name="gpt-5.1"
    )
    
    # Knowledge Base 作成 (Blob Storage ベース)
    blob_knowledge_base = KnowledgeBase(
        name=BLOB_KNOWLEDGE_BASE_NAME,
        description="Blob Storage ベース Knowledge Base",
        retrieval_instructions="が ナレッジベースは Blob Storageの ドキュメントを 含むして あります.",
        answer_instructions="検索された ドキュメントを ベースで 明確な 回答を 提供してください.",
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        knowledge_sources=[
            KnowledgeSourceReference(name=BLOB_KNOWLEDGE_SOURCE_NAME)
        ],
        models=[
            KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_model_params)
        ]
    )
    
    # Knowledge Base 作成
    result_blob_kb = index_client.create_or_update_knowledge_base(blob_knowledge_base)
    
    print(f"\n✅ Knowledge Base 作成 完了!")
    print(f"   - 名前: {result_blob_kb.name}")
    print(f"   - Output Mode: {result_blob_kb.output_mode}")
    print(f"   - Knowledge Source: {BLOB_KNOWLEDGE_SOURCE_NAME}")
    print(f"   - モデル: gpt-5.1")
    
except Exception as e:
    import traceback
    print(f"\n⚠️ Knowledge Base 作成 失敗: {e}")
    print("\n詳細 エラー:")
    traceback.print_exc()
    print("\n💡 解決 方法:")
    print("   1. Knowledge Sourceが まず 作成なったはない 確認")
    print("   2. が未 存在するは 場合 異なる 名前 使用")
    print("   3. Chat モデル 名前 確認")

### 4. KnowledgeAgent2 作成 (Blob Storage ベース)

In [ ]:
KNOWLEDGE_AGENT_2_NAME = "KnowledgeAgent2"

print("KnowledgeAgent2 作成 中...")
print(f"Knowledge Base: {BLOB_KNOWLEDGE_BASE_NAME}")

try:
    # MCP Tool 定の (Blob Storage ベース Knowledge Base)
    mcp_blob_kb_tool = MCPTool(
        server_label="knowledge-base-blob",
        server_url=MCP_ENDPOINT,
        require_approval="never",
        allowed_tools=["knowledge_base_retrieve"],
        project_connection_id=PROJECT_CONNECTION_NAME
    )
    
    # Agent 作成
    blob_knowledge_agent = project_client.agents.create_version(
        agent_name=KNOWLEDGE_AGENT_2_NAME,
        definition=PromptAgentDefinition(
            model="model-router",
            instructions=KNOWLEDGE_INSTRUCTIONS,
            tools=[mcp_blob_kb_tool]
        )
    )
    
    print(f"\n✅ {KNOWLEDGE_AGENT_2_NAME} 作成 完了!")
    print(f"   - Agent: {KNOWLEDGE_AGENT_2_NAME}")
    print(f"   - Model: model-router")
    print(f"   - Knowledge Base: {BLOB_KNOWLEDGE_BASE_NAME}")
    print(f"   - データ ソース: Blob Storage")
    
except Exception as e:
    import traceback
    print(f"\n⚠️ Agent 作成 失敗: {e}")
    print("\n詳細 エラー:")
    traceback.print_exc()
    print("\n💡 解決 方法:")
    print("   1. Knowledge Baseが まず 作成なったはない 確認")
    print("   2. MCP endpoint 確認")
    print("   3. PROJECT_CONNECTION_NAMEが 正しいか 確認")

### 5. KnowledgeAgent2 テスト

In [ ]:
import time
import json

print(f"{'=' * 80}")
print(f"KnowledgeAgent2 テスト (Blob Storage ベース)")
print(f"{'=' * 80}\n")

try:
    # OpenAI client が取得
    openai_client = project_client.get_openai_client()
    
    for i, question in enumerate(test_questions, 1):
        print(f"\n[質問 {i}]: {question}")
        print("-" * 80)
        
        # Conversation 作成
        conversation = openai_client.conversations.create()
        
        # Agentを を通じて レスポンス 作成 (Rate limit 再試も で直 含む)
        print(f"  レスポンス 作成 中...", flush=True)
        
        max_retries = 3
        retry_delay = 15  # 秒
        response = None
        
        for attempt in range(max_retries):
            try:
                response = openai_client.responses.create(
                    conversation=conversation.id,
                    input=question,
                    extra_body={"agent": {"name": blob_knowledge_agent.name, "type": "agent_reference"}}
                )
                print(f"  ✅ レスポンス 作成 成功!")
                break  # 成功すると ループ 終了
                
            except Exception as e:
                error_message = str(e)
                if "429" in error_message or "Too Many Requests" in error_message:
                    if attempt < max_retries - 1:
                        wait_time = retry_delay * (attempt + 1)
                        print(f"  ⏳ Rate limit も月. {wait_time}秒 待機 後 再試も... ({attempt + 1}/{max_retries})")
                        time.sleep(wait_time)
                    else:
                        print(f"  ❌ {max_retries}回 再試も 後にも 失敗.")
                        print(f"  💡 5分 定も 待った 後 再度 時もしてください.")
                        raise
                else:
                    # 異なる エラーは もので throw
                    raise
        
        if response is None:
            print("⚠️ レスポンスを 受けな できなしました。")
            continue
        
        # DEBUG: Response 状態 確認
        print(f"\n🔍 Response 状態: {response.status}")
        print(f"🔍 Response ID: {response.id}")
        
        if hasattr(response, 'output') and isinstance(response.output, list):
            print(f"🔍 Output 長さが: {len(response.output)}")
            for idx, item in enumerate(response.output):
                print(f"\n  Item {idx}: {item.__class__.__name__}")
                if hasattr(item, 'type'):
                    print(f"    type: {item.type}")
                if hasattr(item, 'id'):
                    print(f"    id: {item.id}")
                if hasattr(item, 'text'):
                    print(f"    text (最初 100者): {item.text[:100]}")
                if item.__class__.__name__ == 'McpApprovalRequest':
                    print(f"    ⚠️ Approval 必要!")
                    print(f"    name: {item.name if hasattr(item, 'name') else 'N/A'}")
                    print(f"    arguments: {item.arguments[:200] if hasattr(item, 'arguments') else 'N/A'}")
        
        # レスポンス テキスト 抽出
        actual_text = ""
        
        if hasattr(response, 'output') and isinstance(response.output, list):
            for item in response.output:
                if hasattr(item, 'text'):
                    actual_text = item.text
                    break
        
        print(f"\n[レスポンス]: {actual_text if actual_text else '⚠️ レスポンスを 見つを 数 ありません'}\n")
    
    print("=" * 80)
    print("✅ KnowledgeAgent2 テスト 完了!")
    
except Exception as e:
    import traceback
    print(f"\n⚠️ テスト 失敗: {e}")
    print("\n詳細 エラー:")
    traceback.print_exc()


### 📝 2つの 方式 比較

| 特徴 | AI Search Index | Blob Storage 直接 接続 |
|------|----------------|----------------------|
| **設定 複雑も** | 高い (Import Wizard 必要) | 低い (シンプルな 設定) |
| **自動 更新** | 手動 再インデキシング 必要 | 自動 検出 および インデキシング |
| **カスタマが徴** | 高い (フィールド, スキーマ など) | 低い (自動 構成) |
| **パフォーマンス** | 高い (最適化 が能) | 中間 |
| **使用 事例** | 複雑な 検索 要件 | シンプルな ドキュメント 検索 |

**推奨 事項:**
- 複雑な 検索 および フィルターリングが 必要な 場合 → AI Search Index
- 迅速な プでトタイプ および シンプルな ドキュメント 検索 → Blob Storage

## 📚 追加リソース

- [Foundry IQ 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/knowledge-retrieval?view=foundry&tabs=foundry%2Cpython)
- [Azure AI Search ドキュメント](https://learn.microsoft.com/en-us/azure/search/)
- [RAG パターン ガイド](https://learn.microsoft.com/en-us/azure/search/retrieval-augmented-generation-overview?tabs=docs)
- [ベクトル 検索 最適化](https://learn.microsoft.com/en-us/azure/search/vector-search-overview)